# 01 — Mars Cleaning & Preprocessing (executed)
**NASA Space Apps — Mars terrain and hypothetical illumination**

The source HiRISE/PDS3 and MOLA/PDS3 IMG/LBL products are inspected without altering the raw files. The pipeline selects two **disconnected** patches, retains native 2m cells and NoData, derives an approximate 30m analysis DEM, computes physically defined terrain metrics and explicitly finite-range cast shadows, and saves GeoTIFF/CSV/JSON/PNG outputs. Full streaming QA is performed on all raw HiRISE and MOLA pixels.

**The code below actually runs the full pipeline.** Work deliberately happens in a reusable Python script (`pipeline.py`) for testability, block streaming and use outside Jupyter. The notebook records return status and results. Official references are in `README.md` and `metadata/processing_parameters.json`. This is not a claim of full scientific calibration.


In [1]:
from pathlib import Path
import sys, subprocess, json
here = Path.cwd()
assert (here/'pipeline.py').exists(), f'Run notebook with working directory Mars_Processed, currently {here}'
print('Mode:', 'full (PDS sources found)' if (here.parent/'Mars').exists() else 'derive (sources absent: rebuilding DEM-derived products from the stored 30 m analysis DEMs)')
print('Python:',sys.version.split()[0]);print('Input images:', [p.name for p in (here.parent/'Mars').glob('*.IMG')])

Python: 3.13.5
Input images: ['DTEED_042328_1985_033150_1985_A01.IMG', 'DTEED_033651_1990_033717_1990_A01.IMG']


## 1. Fully execute streaming QA, 30m preprocessing, shadows and verification
The script performs actual full-source windowed HiRISE scans, strict big-endian PDS-labeled MOLA parsing, independent patch selection and complete product writing. It does **not** fill sinks, smooth, replace NoData with zero or remove negative elevation. The shadow routine uses 1200m finite horizon and flags unknown rays, not unsupported full-visibility claims.

In [2]:
result = subprocess.run([sys.executable, str(here/'pipeline.py')],cwd=here,text=True,capture_output=True,check=False)
print(result.stdout)
if result.returncode:
 print(result.stderr[-3500:])
 raise RuntimeError('Full pipeline failed with exit code '+str(result.returncode))
print('Pipeline return code:',result.returncode)

1/6 Native sources inspection + hashes
2/6 MOLA strict binary parse + independent context
3/6 Synthetic slope/cast-shadow tests
4/6 HiRISE independent patch aggregation, terrain and 48 casts
   DTEED_033651_1990_033717_1990_A01 native valid% 100.0 30m valid% 100.0
   DTEED_042328_1985_033150_1985_A01 native valid% 100.0 30m valid% 100.0
5/6 Reports, sources, frontend, verification
6/6 Final {"tests": 11, "test_passed": true, "reopened_geotiffs": 168, "output_files": 196, "elapsed_seconds": 7.3}

Pipeline return code: 0


## 2. Check the actual input quality report
HiRISE invalid floats and >50m adjacent same-block jumps are **diagnostic flags**, not automatic removal. The 2m patch can be clean even if the complete raw footprint contains voids.

In [3]:
before=json.loads((here/'reports/input_quality_before.json').read_text())
for x in before:
 print(x['source'], 'source valid %',round(x['valid_pct'],3),'elevation range (m)',round(x['elevation_min_m'],2),round(x['elevation_max_m'],2),'jumps flagged:',x['adjacent_jumps_gt_50m_within_blocks'])
mola=json.loads((here/'metadata/MOLA_metadata.json').read_text())
print('MOLA raw min,max (m):',mola['minimum_m'],mola['maximum_m'],'total samples:',mola['count'])

DTEED_033651_1990_033717_1990_A01.IMG source valid % 75.754 elevation range (m) -2740.42 -2152.6 jumps flagged: 161
DTEED_042328_1985_033150_1985_A01.IMG source valid % 69.389 elevation range (m) -1914.45 -456.63 jumps flagged: 0
MOLA raw min,max (m): -5062 3210 total samples: 64880640


## 3. Inspect patch coverage and terrain metrics
Every derived GeoTIFF has numerical data. Roughness is full-valid-window standard deviation; local relief is local max minus min. A 30m grid is **not evidence of equal effective resolution** versus NASADEM.

In [4]:
after=json.loads((here/'reports/quality_after.json').read_text())
for x in after['patches']:
 print(x['source'], 'native valid %',round(x['actual_native_valid_pct'],2), '30m valid area km²',round(x['valid_analysis_area_km2'],3),'slope deg mean',round(x['terrain_stats']['slope']['mean'],3))
assert after['all_native_modified_cells']==0

DTEED_033651_1990_033717_1990_A01 native valid % 100.0 30m valid area km² 7.339 slope deg mean 8.723
DTEED_042328_1985_033150_1985_A01 native valid % 100.0 30m valid area km² 7.34 slope deg mean 8.577


## 4. Shadow behavior and synthetic tests
Cast shadow: `1` obstructed, `0` clear within **1.2 km sampled rays**, `255` unresolved. Self-shadow and normal-incidence cosine are separately stored. Sun angles are hypothetical azimuths clockwise **from local raster grid north**, and are not real date/time ephemerides.

In [5]:
tests=json.loads((here/'reports/verification_tests.json').read_text())
reopen=json.loads((here/'reports/geotiff_reopen_validation.json').read_text())
assert tests['all_passed'] and reopen['valid']
print('Synthetic terrain/shadow tests:',len(tests['tests']),'passed:',sum(x['passed'] for x in tests['tests']))
print('Output GeoTIFF products reopened:',reopen['reopened_geotiffs'])
import csv
for report in sorted((here/'reports').glob('*sun_summary.csv')):
 rows=list(csv.DictReader(report.open()))
 print(report.name, 'scenarios:',len(rows),'example first solar scenario counts:', {k:rows[0][k] for k in ('cast_shadow_cells','cast_clear_cells','cast_unresolved_cells')})

Synthetic terrain/shadow tests: 11 passed: 11
Output GeoTIFF products reopened: 168
DTEED_033651_1990_033717_1990_A01_sun_summary.csv scenarios: 24 example first solar scenario counts: {'cast_shadow_cells': '2728', 'cast_clear_cells': '3552', 'cast_unresolved_cells': '1901'}
DTEED_042328_1985_033150_1985_A01_sun_summary.csv scenarios: 24 example first solar scenario counts: {'cast_shadow_cells': '288', 'cast_clear_cells': '4653', 'cast_unresolved_cells': '3240'}


## 5. Evidence, limits, frontend delivery
MOLA and HiRISE are not artificially merged: the **Mars 2000 datum** description in the HiRISE PDS header and **GMM3 areoid / IAU2000** description in the MOLA label must be reconciled for absolute vertical comparison. Data are not available for Earth side, so there is no cross-planet similarity ranking. The frontend gets Mars-projected projected polygons; they must **never be treated as Earth WGS84 GeoJSON**. Inspect `manifest.csv` and `README.md` for all products. This notebook's execution and tests are a reproducibility record, not external planetary science accreditation.

In [6]:
assert (here/'frontend/dashboard_data.json').exists()
assert (here/'processed/common/MOLA_regional_context_128ppd.tif').exists()
print('Notebook workflow completed successfully; source files remain unchanged.')

Notebook workflow completed successfully; source files remain unchanged.
